In [1]:
import os, random, time, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.vim import VMamba, MambaConfig
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
AUG_ROOT = "D:/mamba_model/aug_clean_tio"       
TAG      = "tio_cnn_scratch"                                  
# ───────────────────────────────────────────────────────────────

COHORT_CSV = "D:/mamba_model/thesis_cohort_clean.csv"
MRI_CACHE  = f"{AUG_ROOT}/roi_mri"
PET_CACHE  = f"{AUG_ROOT}/roi_pet"
CKPT_DIR   = f"D:/mamba_model/checkpoints_v7_roi_{TAG}"
RESULTS    = f"D:/mamba_model/v7_roi_{TAG}_results.json"
os.makedirs(CKPT_DIR, exist_ok=True)

SPLIT_SEED  = 42
AUG_SEEDS   = [1, 101, 42]
BATCH_SIZE  = 4
NUM_WORKERS = 0          

print(f"aug:  {AUG_ROOT}")
print(f"ckpt: {CKPT_DIR}")
for p in (MRI_CACHE, PET_CACHE):
    n = len(os.listdir(p)) if os.path.isdir(p) else 0
    print(f"  {os.path.basename(p)}: {n} files{'  *** MISSING ***' if n == 0 else ''}")

aug:  D:/mamba_model/aug_clean_tio
ckpt: D:/mamba_model/checkpoints_v7_roi_tio_cnn_scratch
  roi_mri: 560 files
  roi_pet: 560 files


In [3]:
class VimEncoder(nn.Module):
    """Bidirectional Mamba over a token sequence."""
    def __init__(self, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        cfg = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state,
                          bidirectional=True, divide_output=True,
                          pscan=True, use_cuda=False)
        self.encoder = VMamba(cfg)
        self.final_norm = nn.LayerNorm(d_model)
    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))

In [4]:
class ScratchCNNEncoder(nn.Module):
    """From-scratch shallow CNN: one token per ROI (6 tokens, not 3072)."""
    def __init__(self, n_rois=6, d_model=32):
        super().__init__()
        self.conv1 = nn.Conv3d(1, 16, 5, stride=4, padding=2)
        self.conv2 = nn.Conv3d(16, 32, 3, stride=2, padding=1)
        self.conv3 = nn.Conv3d(32, d_model, 3, stride=2, padding=1)
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.roi_embed = nn.Embedding(n_rois, d_model)
        with torch.no_grad():
            self.roi_embed.weight.mul_(0.02)
    def forward(self, rois):
        B, n = rois.shape[:2]
        x = rois.reshape(B * n, 1, *rois.shape[-3:])
        x = F.relu(self.conv1(x)); x = F.relu(self.conv2(x)); x = F.relu(self.conv3(x))
        x = self.pool(x).flatten(1)
        return x.reshape(B, n, -1) + self.roi_embed.weight[None]


class CNNMambaBranch(nn.Module):
    def __init__(self, encoder_cls, n_rois=6, d_model=32, n_layers=2, **kw):
        super().__init__()
        self.cnn = encoder_cls(n_rois=n_rois, d_model=d_model, **kw)
        self.vim = VimEncoder(d_model, n_layers)
    def forward(self, rois):
        return self.vim(self.cnn(rois)).mean(dim=1)


def make_cnn_models(encoder_cls, **enc_kw):
    class _Uni(nn.Module):
        def __init__(self, n_rois=6, d_model=32, n_layers=2,
                     n_classes=2, d_state=16, dropout=0.4, **_):
            super().__init__()
            self.branch = CNNMambaBranch(encoder_cls, n_rois, d_model, n_layers, **enc_kw)
            self.dropout = nn.Dropout(dropout)
            self.classifier = nn.Linear(d_model, n_classes)
        def forward(self, rois):
            return self.classifier(self.dropout(self.branch(rois)))

    class _MM(nn.Module):
        def __init__(self, n_rois=6, d_model=32, n_layers=2,
                     n_classes=2, d_state=16, dropout=0.4, **_):
            super().__init__()
            self.mri_branch = CNNMambaBranch(encoder_cls, n_rois, d_model, n_layers, **enc_kw)
            self.pet_branch = CNNMambaBranch(encoder_cls, n_rois, d_model, n_layers, **enc_kw)
            self.dropout = nn.Dropout(dropout)
            self.classifier = nn.Linear(d_model * 2, n_classes)
        def forward(self, mri, pet):
            f = torch.cat([self.mri_branch(mri), self.pet_branch(pet)], dim=1)
            return self.classifier(self.dropout(f))
    return _Uni, _MM


VisionMambaModel, MultimodalVisionMambaModel = make_cnn_models(ScratchCNNEncoder)
print("CNN(scratch) + Mamba, 6 tokens per modality")

CNN(scratch) + Mamba, 6 tokens per modality


In [5]:
df = pd.read_csv(COHORT_CSV)
sessions, labels = df["mri_session"].values, df["outcome_label"].values

X_tv, X_test, y_tv, y_test = train_test_split(
    sessions, labels, test_size=0.2, random_state=SPLIT_SEED, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=SPLIT_SEED, stratify=y_tv)

session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))
print(f"train {len(X_train)} | val {len(X_val)} | test {len(X_test)} "
      f"| test pos {int(y_test.sum())}")


_CACHE = {}

def load_cached(path):
    a = _CACHE.get(path)
    if a is None:
        a = np.load(path).astype(np.float32)
        _CACHE[path] = a
    return a


class ROIDataset(Dataset):
    """Single modality. Training set includes 3 augmented copies per subject."""
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples, self.cache_dir = [], cache_dir
        for ses, lab in zip(sessions, labels):
            key = ses if is_mri else session_to_subject[ses]
            self.samples.append((key, lab, "orig"))
            if is_train:
                for s in AUG_SEEDS:
                    self.samples.append((key, lab, f"aug{s}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        key, lab, ver = self.samples[i]
        a = load_cached(f"{self.cache_dir}/{key}_{ver}.npy")
        return torch.from_numpy(a).unsqueeze(1), torch.tensor(lab, dtype=torch.long), key


class MultimodalROIDataset(Dataset):
    """Pairs MRI and PET for the same subject and the same augmentation seed."""
    def __init__(self, sessions, labels, mri_dir, pet_dir, is_train=False):
        self.samples, self.mri_dir, self.pet_dir = [], mri_dir, pet_dir
        for ses, lab in zip(sessions, labels):
            sid = session_to_subject[ses]
            self.samples.append((ses, sid, lab, "orig"))
            if is_train:
                for s in AUG_SEEDS:
                    self.samples.append((ses, sid, lab, f"aug{s}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        mk, pk, lab, ver = self.samples[i]
        m = load_cached(f"{self.mri_dir}/{mk}_{ver}.npy")
        p = load_cached(f"{self.pet_dir}/{pk}_{ver}.npy")
        return (torch.from_numpy(m).unsqueeze(1), torch.from_numpy(p).unsqueeze(1),
                torch.tensor(lab, dtype=torch.long), mk)


def dl(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=(NUM_WORKERS > 0))

mri_loaders = (dl(ROIDataset(X_train, y_train, MRI_CACHE, True, True), True),
               dl(ROIDataset(X_val,   y_val,   MRI_CACHE, True, False)),
               dl(ROIDataset(X_test,  y_test,  MRI_CACHE, True, False)))

pet_loaders = (dl(ROIDataset(X_train, y_train, PET_CACHE, False, True), True),
               dl(ROIDataset(X_val,   y_val,   PET_CACHE, False, False)),
               dl(ROIDataset(X_test,  y_test,  PET_CACHE, False, False)))

mm_loaders  = (dl(MultimodalROIDataset(X_train, y_train, MRI_CACHE, PET_CACHE, True), True),
               dl(MultimodalROIDataset(X_val,   y_val,   MRI_CACHE, PET_CACHE, False)),
               dl(MultimodalROIDataset(X_test,  y_test,  MRI_CACHE, PET_CACHE, False)))

print(f"train samples (4x augmented): {len(mri_loaders[0].dataset)}")

# first pass fills the cache from disk, second should be near-instant
t0 = time.time()
for i, _ in enumerate(mri_loaders[0]):
    if i >= 20: break
cold = time.time() - t0

t0 = time.time()
for i, _ in enumerate(mri_loaders[0]):
    if i >= 20: break
warm = time.time() - t0

print(f"20 batches: cold {cold:.1f}s -> warm {warm:.1f}s")
print(f"cached arrays: {len(_CACHE)}  (~{sum(a.nbytes for a in _CACHE.values())/1e9:.1f} GB)")

train 120 | val 40 | test 40 | test pos 20
train samples (4x augmented): 480
20 batches: cold 2.1s -> warm 2.3s
cached arrays: 152  (~1.0 GB)


In [6]:
def train_epoch(model, loader, opt, crit, mm):
    model.train(); tot = 0
    for batch in loader:
        opt.zero_grad()
        if mm:
            a, b, lb, _ = batch; out = model(a.to(device), b.to(device))
        else:
            a, lb, _ = batch;    out = model(a.to(device))
        loss = crit(out, lb.to(device)); loss.backward(); opt.step(); tot += loss.item()
    return tot / len(loader)

def evaluate(model, loader, crit, mm):
    model.eval(); tot, P, L = 0, [], []
    with torch.no_grad():
        for batch in loader:
            if mm:
                a, b, lb, _ = batch; out = model(a.to(device), b.to(device))
            else:
                a, lb, _ = batch;    out = model(a.to(device))
            tot += crit(out, lb.to(device)).item()
            P.extend(out.argmax(1).cpu().numpy()); L.extend(lb.numpy())
    return (tot/len(loader), np.mean(np.array(P) == np.array(L)),
            recall_score(L, P, zero_division=0), specificity_score(L, P))

def measure_inference(model, loader, mm, n=20):
    model.eval(); ts = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n: break
            if mm:
                a, b = batch[0].to(device), batch[1].to(device); bs = a.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time(); _ = model(a, b)
            else:
                a = batch[0].to(device); bs = a.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time(); _ = model(a)
            if device.type == 'cuda': torch.cuda.synchronize()
            ts.append((time.time() - t0) / bs)
    return np.mean(ts), np.std(ts)

def compute_flops(model, loader, mm):
    try:
        model.eval(); b = next(iter(loader))
        with torch.no_grad():
            inp = (b[0][:1].to(device), b[1][:1].to(device)) if mm else (b[0][:1].to(device),)
            macs, _ = profile(model, inputs=inp, verbose=False)
        return macs * 2
    except Exception as e:
        print(f"  (FLOPs failed: {e})"); return None


def run_seed(seed, model_cls, loaders, mm, prefix,
             max_epochs=101, patience=15, min_epochs=25, lr=1e-4, log_every=5):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    np.random.seed(seed); random.seed(seed)
    tr, va, te = loaders

    model = model_cls(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    crit  = nn.CrossEntropyLoss(label_smoothing=0.05)
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sch   = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)

    best, no_imp, best_ep, total = float('inf'), 0, 0, 0
    path = f"{CKPT_DIR}/{prefix}_seed{seed}.pt"
    print(f"\n--- {prefix} seed {seed} ---")

    for ep in range(1, max_epochs):
        t0 = time.time()
        trl = train_epoch(model, tr, opt, crit, mm)
        vl, vacc, vtpr, vtnr = evaluate(model, va, crit, mm)
        sch.step(vl); dt = time.time() - t0; total += dt
        if ep % log_every == 0 or ep == 1:
            print(f"  ep {ep:>3} | train {trl:.4f} | val {vl:.4f} | "
                  f"acc {vacc:.3f} tpr {vtpr:.3f} tnr {vtnr:.3f} | {dt:.0f}s")
        if vl < best:
            best, best_ep, no_imp = vl, ep, 0
            torch.save(model.state_dict(), path)
        else:
            no_imp += 1
            if ep >= min_epochs and no_imp >= patience:
                print(f"  early stop {ep}, best {best_ep}"); break

    if best_ep < 5:
        print(f"  WARNING: best epoch {best_ep} -- may not have trained")

    model.load_state_dict(torch.load(path, weights_only=True))
    _, acc, tpr, tnr = evaluate(model, te, crit, mm)
    npar = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_m, inf_s = measure_inference(model, te, mm)
    fl = compute_flops(model, te, mm)

    print(f"  >>> TEST Acc={acc*100:.1f}% TPR={tpr*100:.1f}% TNR={tnr*100:.1f}% | "
          f"params={npar:,} train={total/60:.1f}min inf={inf_m*1000:.2f}ms "
          f"{f'{fl/1e9:.2f}GFLOPs' if fl else ''} best_ep={best_ep}")

    return {"seed": seed, "acc": acc, "tpr": tpr, "tnr": tnr, "best_epoch": best_ep,
            "train_time_sec": total, "n_params": npar,
            "inf_time_ms": inf_m*1000, "flops": fl}

results = {"mri": [], "pet": [], "mm": []}

In [7]:
results["mri"].append(run_seed(1, VisionMambaModel, mri_loaders, False, "v7_roi_cnns_mri"))


--- v7_roi_cnns_mri seed 1 ---
  ep   1 | train 0.7125 | val 0.6915 | acc 0.500 tpr 1.000 tnr 0.000 | 23s
  ep   5 | train 0.7014 | val 0.6873 | acc 0.675 tpr 0.600 tnr 0.750 | 3s
  ep  10 | train 0.6226 | val 0.6403 | acc 0.650 tpr 0.800 tnr 0.500 | 2s
  ep  15 | train 0.4020 | val 0.7269 | acc 0.625 tpr 0.600 tnr 0.650 | 2s
  ep  20 | train 0.1437 | val 0.9593 | acc 0.650 tpr 0.650 tnr 0.650 | 2s
  ep  25 | train 0.1262 | val 1.0393 | acc 0.650 tpr 0.650 tnr 0.650 | 2s
  early stop 25, best 10
  >>> TEST Acc=67.5% TPR=80.0% TNR=55.0% | params=71,330 train=1.3min inf=2.20ms 0.20GFLOPs best_ep=10


In [8]:
results["mri"].append(run_seed(7, VisionMambaModel, mri_loaders, False, "v7_roi_cnns_mri"))


--- v7_roi_cnns_mri seed 7 ---
  ep   1 | train 0.7175 | val 0.7038 | acc 0.500 tpr 1.000 tnr 0.000 | 3s
  ep   5 | train 0.6851 | val 0.6732 | acc 0.600 tpr 0.450 tnr 0.750 | 2s
  ep  10 | train 0.5140 | val 0.6689 | acc 0.675 tpr 0.700 tnr 0.650 | 2s
  ep  15 | train 0.3355 | val 0.7622 | acc 0.675 tpr 0.700 tnr 0.650 | 2s
  ep  20 | train 0.1407 | val 0.9544 | acc 0.600 tpr 0.700 tnr 0.500 | 2s
  ep  25 | train 0.1258 | val 1.0640 | acc 0.675 tpr 0.550 tnr 0.800 | 2s
  early stop 25, best 9
  >>> TEST Acc=67.5% TPR=70.0% TNR=65.0% | params=71,330 train=1.0min inf=1.16ms 0.20GFLOPs best_ep=9


In [9]:
results["mri"].append(run_seed(123, VisionMambaModel, mri_loaders, False, "v7_roi_cnns_mri"))


--- v7_roi_cnns_mri seed 123 ---
  ep   1 | train 0.7288 | val 0.6925 | acc 0.500 tpr 1.000 tnr 0.000 | 2s
  ep   5 | train 0.6915 | val 0.6963 | acc 0.500 tpr 1.000 tnr 0.000 | 2s
  ep  10 | train 0.6294 | val 0.6247 | acc 0.700 tpr 0.700 tnr 0.700 | 2s
  ep  15 | train 0.3521 | val 0.6800 | acc 0.700 tpr 0.600 tnr 0.800 | 2s
  ep  20 | train 0.1545 | val 0.8872 | acc 0.600 tpr 0.500 tnr 0.700 | 2s
  ep  25 | train 0.1264 | val 1.0954 | acc 0.575 tpr 0.450 tnr 0.700 | 2s
  early stop 27, best 12
  >>> TEST Acc=65.0% TPR=75.0% TNR=55.0% | params=71,330 train=1.0min inf=1.16ms 0.20GFLOPs best_ep=12


In [10]:
results["pet"].append(run_seed(1, VisionMambaModel, pet_loaders, False, "v7_roi_cnns_pet"))


--- v7_roi_cnns_pet seed 1 ---
  ep   1 | train 0.7147 | val 0.6911 | acc 0.475 tpr 0.950 tnr 0.000 | 33s
  ep   5 | train 0.6992 | val 0.6822 | acc 0.625 tpr 0.350 tnr 0.900 | 2s
  ep  10 | train 0.5834 | val 0.6044 | acc 0.750 tpr 0.550 tnr 0.950 | 3s
  ep  15 | train 0.2730 | val 0.8485 | acc 0.700 tpr 0.500 tnr 0.900 | 2s
  ep  20 | train 0.1262 | val 1.0724 | acc 0.650 tpr 0.650 tnr 0.650 | 3s
  ep  25 | train 0.1261 | val 1.1575 | acc 0.625 tpr 0.650 tnr 0.600 | 2s
  early stop 25, best 9
  >>> TEST Acc=65.0% TPR=50.0% TNR=80.0% | params=71,330 train=1.6min inf=2.62ms 0.20GFLOPs best_ep=9


In [11]:
results["pet"].append(run_seed(7, VisionMambaModel, pet_loaders, False, "v7_roi_cnns_pet"))


--- v7_roi_cnns_pet seed 7 ---
  ep   1 | train 0.7160 | val 0.6972 | acc 0.500 tpr 1.000 tnr 0.000 | 3s
  ep   5 | train 0.6820 | val 0.6575 | acc 0.625 tpr 0.250 tnr 1.000 | 2s
  ep  10 | train 0.5968 | val 0.5838 | acc 0.700 tpr 0.450 tnr 0.950 | 2s
  ep  15 | train 0.3420 | val 0.6482 | acc 0.725 tpr 0.750 tnr 0.700 | 2s
  ep  20 | train 0.1344 | val 0.7734 | acc 0.700 tpr 0.700 tnr 0.700 | 2s
  ep  25 | train 0.1247 | val 0.8029 | acc 0.725 tpr 0.700 tnr 0.750 | 2s
  early stop 26, best 11
  >>> TEST Acc=60.0% TPR=50.0% TNR=70.0% | params=71,330 train=1.1min inf=1.13ms 0.20GFLOPs best_ep=11


In [12]:
results["pet"].append(run_seed(123, VisionMambaModel, pet_loaders, False, "v7_roi_cnns_pet"))


--- v7_roi_cnns_pet seed 123 ---
  ep   1 | train 0.7281 | val 0.6924 | acc 0.500 tpr 1.000 tnr 0.000 | 2s
  ep   5 | train 0.6873 | val 0.6806 | acc 0.500 tpr 1.000 tnr 0.000 | 2s
  ep  10 | train 0.6110 | val 0.5900 | acc 0.775 tpr 0.750 tnr 0.800 | 2s
  ep  15 | train 0.3168 | val 0.5718 | acc 0.750 tpr 0.700 tnr 0.800 | 2s
  ep  20 | train 0.1299 | val 0.7922 | acc 0.725 tpr 0.600 tnr 0.850 | 2s
  ep  25 | train 0.1251 | val 0.7395 | acc 0.650 tpr 0.600 tnr 0.700 | 2s
  early stop 27, best 12
  >>> TEST Acc=67.5% TPR=70.0% TNR=65.0% | params=71,330 train=1.1min inf=1.11ms 0.20GFLOPs best_ep=12


In [13]:
results["mm"].append(run_seed(1, MultimodalVisionMambaModel, mm_loaders, True, "v7_roi_cnns_mm"))


--- v7_roi_cnns_mm seed 1 ---
  ep   1 | train 0.7237 | val 0.6970 | acc 0.500 tpr 0.000 tnr 1.000 | 7s
  ep   5 | train 0.6826 | val 0.6738 | acc 0.525 tpr 1.000 tnr 0.050 | 7s
  ep  10 | train 0.5643 | val 0.5771 | acc 0.675 tpr 0.750 tnr 0.600 | 5s
  ep  15 | train 0.2278 | val 0.6698 | acc 0.650 tpr 0.700 tnr 0.600 | 5s
  ep  20 | train 0.1300 | val 0.7355 | acc 0.700 tpr 0.600 tnr 0.800 | 5s
  ep  25 | train 0.1223 | val 0.8255 | acc 0.650 tpr 0.500 tnr 0.800 | 5s
  early stop 27, best 12
  >>> TEST Acc=67.5% TPR=70.0% TNR=65.0% | params=142,658 train=2.3min inf=12.94ms 0.41GFLOPs best_ep=12


In [14]:
results["mm"].append(run_seed(7, MultimodalVisionMambaModel, mm_loaders, True, "v7_roi_cnns_mm"))


--- v7_roi_cnns_mm seed 7 ---
  ep   1 | train 0.7254 | val 0.6912 | acc 0.500 tpr 1.000 tnr 0.000 | 6s
  ep   5 | train 0.6947 | val 0.6757 | acc 0.725 tpr 0.900 tnr 0.550 | 5s
  ep  10 | train 0.5604 | val 0.5632 | acc 0.725 tpr 0.850 tnr 0.600 | 6s
  ep  15 | train 0.2182 | val 0.6961 | acc 0.675 tpr 0.750 tnr 0.600 | 4s
  ep  20 | train 0.1263 | val 0.8712 | acc 0.675 tpr 0.600 tnr 0.750 | 4s
  ep  25 | train 0.1235 | val 0.8753 | acc 0.650 tpr 0.600 tnr 0.700 | 5s
  early stop 26, best 11
  >>> TEST Acc=65.0% TPR=60.0% TNR=70.0% | params=142,658 train=2.1min inf=5.26ms 0.41GFLOPs best_ep=11


In [15]:
results["mm"].append(run_seed(123, MultimodalVisionMambaModel, mm_loaders, True, "v7_roi_cnns_mm"))


--- v7_roi_cnns_mm seed 123 ---
  ep   1 | train 0.7153 | val 0.7104 | acc 0.500 tpr 0.000 tnr 1.000 | 5s
  ep   5 | train 0.6900 | val 0.6595 | acc 0.675 tpr 0.400 tnr 0.950 | 5s
  ep  10 | train 0.5303 | val 0.5469 | acc 0.700 tpr 0.800 tnr 0.600 | 5s
  ep  15 | train 0.2065 | val 0.5939 | acc 0.700 tpr 0.600 tnr 0.800 | 5s
  ep  20 | train 0.1301 | val 0.7056 | acc 0.650 tpr 0.600 tnr 0.700 | 5s
  ep  25 | train 0.1241 | val 0.7133 | acc 0.725 tpr 0.750 tnr 0.700 | 4s
  early stop 25, best 10
  >>> TEST Acc=72.5% TPR=75.0% TNR=70.0% | params=142,658 train=1.9min inf=2.13ms 0.41GFLOPs best_ep=10


In [16]:
INCLUDE_SEEDS = [1, 7, 123]

def summarize(rs, name, include=INCLUDE_SEEDS):
    rs = [r for r in rs if r['seed'] in include]
    if not rs: print(f"{name}: no runs"); return
    a = [r['acc'] for r in rs]; t = [r['tpr'] for r in rs]; n = [r['tnr'] for r in rs]
    tm = [r['train_time_sec'] for r in rs]; inf = [r['inf_time_ms'] for r in rs]
    fl = [r['flops'] for r in rs if r['flops']]
    sd = (lambda v: np.std(v, ddof=1)*100 if len(v) > 1 else 0.0)
    print(f"{name}: Acc={np.mean(a)*100:.1f}±{sd(a):.1f}% | "
          f"TPR={np.mean(t)*100:.1f}±{sd(t):.1f}% | TNR={np.mean(n)*100:.1f}±{sd(n):.1f}% | "
          f"Params={rs[0]['n_params']:,} | Train={np.mean(tm)/60:.1f}m | "
          f"Inf={np.mean(inf):.2f}ms | {f'{np.mean(fl)/1e9:.2f}GFLOPs' if fl else 'N/A'} "
          f"| seeds={[r['seed'] for r in rs]}")

print(f"=== v7 ROI Vision Mamba — {TAG} augmentation, 200-subject cohort ===")
print(f"    seeds {INCLUDE_SEEDS}\n")
for k, n in [('mri','MRI-only  '), ('pet','PET-only  '), ('mm','Multimodal')]:
    summarize(results[k], n)

with open(RESULTS, 'w') as f:
    json.dump({k: [{kk: (float(vv) if isinstance(vv, (float, np.floating)) else vv)
                    for kk, vv in r.items()} for r in v] for k, v in results.items()},
              f, indent=2)
print(f"\nsaved {RESULTS} (all seeds retained)")

=== v7 ROI Vision Mamba — tio_cnn_scratch augmentation, 200-subject cohort ===
    seeds [1, 7, 123]

MRI-only  : Acc=66.7±1.4% | TPR=75.0±5.0% | TNR=58.3±5.8% | Params=71,330 | Train=1.1m | Inf=1.51ms | 0.20GFLOPs | seeds=[1, 7, 123]
PET-only  : Acc=64.2±3.8% | TPR=56.7±11.5% | TNR=71.7±7.6% | Params=71,330 | Train=1.2m | Inf=1.62ms | 0.20GFLOPs | seeds=[1, 7, 123]
Multimodal: Acc=68.3±3.8% | TPR=68.3±7.6% | TNR=68.3±2.9% | Params=142,658 | Train=2.1m | Inf=6.78ms | 0.41GFLOPs | seeds=[1, 7, 123]

saved D:/mamba_model/v7_roi_tio_cnn_scratch_results.json (all seeds retained)


In [5]:
#  GFLOPs for one forward pass, batch size 1 -- 3D CNN tokenisation

#  Architecture alone determines these figures, so no training, checkpoints
#  or data are needed.
import copy
from torch.utils.flop_counter import FlopCounterMode
from mambapy.vim import VMamba as _VMamba


def count_flops(model, inputs):
    """Returns (conv_linear, scan, n_tokens) for one forward pass."""
    m = copy.deepcopy(model).eval().cpu()
    inputs = [t.detach().cpu() for t in inputs]
    mamba_log, handles = [], []

    def mk(mod):
        def hook(_, inp, __):
            c = mod.config
            mamba_log.append(dict(L=inp[0].shape[1], ed=c.d_inner, n=c.d_state,
                                  layers=c.n_layers,
                                  bi=getattr(c, "bidirectional", False)))
        return hook
    for mod in m.modules():
        if isinstance(mod, _VMamba):
            handles.append(mod.register_forward_hook(mk(mod)))

    counter = FlopCounterMode(display=False)
    with torch.no_grad(), counter:
        m(*inputs)
    for h in handles:
        h.remove()

    cl = counter.get_total_flops()
    sc = sum(6 * r["ed"] * r["n"] * r["L"] * r["layers"] * (2 if r["bi"] else 1)
             for r in mamba_log)
    tokens = mamba_log[0]["L"] if mamba_log else 0
    del m
    return cl, sc, tokens


def report(name, model, inputs):
    p = sum(q.numel() for q in model.parameters() if q.requires_grad)
    cl, sc, tok = count_flops(model, inputs)
    print(f"  {name:28s} {p:>9,}p | {tok} tokens | "
          f"conv+linear {cl/1e9:.4f} + scan {sc/1e9:.4f} = {(cl+sc)/1e9:.4f} GFLOPs")


roi = lambda: torch.randn(1, 6, 1, 64, 64, 64)
print("GFLOPs, one forward pass, batch size 1 -- 3D CNN tokenisation\n")
report("CNN tokenisation (unimodal)",   VisionMambaModel().cpu(),           [roi()])
report("CNN tokenisation (multimodal)", MultimodalVisionMambaModel().cpu(), [roi(), roi()])

GFLOPs, one forward pass, batch size 1 -- 3D CNN tokenisation

  CNN tokenisation (unimodal)     71,330p | 6 tokens | conv+linear 0.2048 + scan 0.0001 = 0.2049 GFLOPs
  CNN tokenisation (multimodal)   142,658p | 6 tokens | conv+linear 0.4096 + scan 0.0003 = 0.4099 GFLOPs
